In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('train.csv', encoding='cp1252')

In [3]:
df.head()

,textID,text,selected_text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Km²),Density (P/Km²)
0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral,morning,0-20,Afghanistan,38928346,652860.0,60
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative,noon,21-30,Albania,2877797,27400.0,105
2,088c60f138,my boss is bullying me...,bullying me,negative,night,31-45,Algeria,43851044,2381740.0,18
3,9642c003ef,what interview! leave me alone,leave me alone,negative,morning,46-60,Andorra,77265,470.0,164
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",negative,noon,60-70,Angola,32866272,1246700.0,26


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27481 entries, 0 to 27480
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   textID            27481 non-null  object 
 1   text              27480 non-null  object 
 2   selected_text     27480 non-null  object 
 3   sentiment         27481 non-null  object 
 4   Time of Tweet     27481 non-null  object 
 5   Age of User       27481 non-null  object 
 6   Country           27481 non-null  object 
 7   Population -2020  27481 non-null  int64  
 8   Land Area (Km²)   27481 non-null  float64
 9   Density (P/Km²)   27481 non-null  int64  
dtypes: float64(1), int64(2), object(7)
memory usage: 2.1+ MB


In [5]:
df.isnull().sum()

textID              0
text                1
selected_text       1
sentiment           0
Time of Tweet       0
Age of User         0
Country             0
Population -2020    0
Land Area (Km²)     0
Density (P/Km²)     0
dtype: int64

In [6]:
df.shape

(27481, 10)

In [7]:
df = df.drop_duplicates()

In [8]:
df = df.drop(['Time of Tweet','selected_text','textID','Country','Population -2020','Land Area (Km²)','Density (P/Km²)'],axis=1)

In [9]:
df.head()

,text,sentiment,Age of User
0,"I`d have responded, if I were going",neutral,0-20
1,Sooo SAD I will miss you here in San Diego!!!,negative,21-30
2,my boss is bullying me...,negative,31-45
3,what interview! leave me alone,negative,46-60
4,"Sons of ****, why couldn`t they put them on t...",negative,60-70


In [10]:
df['Age of User'].unique()

array(['0-20', '21-30', '31-45', '46-60', '60-70', '70-100'], dtype=object)

In [11]:
df.isnull().sum()

text           1
sentiment      0
Age of User    0
dtype: int64

In [12]:
df = df.dropna()

In [13]:
import re
import string

def clean_text(text):
    text = str(text)

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # Remove mentions
    text = re.sub(r'@\w+', '', text)

    # Remove hashtags symbol but keep the word
    text = re.sub(r'#', '', text)

    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text
df['text'] = df['text'].apply(clean_text)

In [14]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

X = df.drop('sentiment',axis=1)
y = le.fit_transform(df['sentiment'])

In [15]:
X.head()

,text,Age of User
0,id have responded if i were going,0-20
1,sooo sad i will miss you here in san diego,21-30
2,my boss is bullying me,31-45
3,what interview leave me alone,46-60
4,sons of why couldnt they put them on the relea...,60-70


In [16]:
pd.Series(y).value_counts()

1    11117
2     8582
0     7781
Name: count, dtype: int64

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [18]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer('all-MiniLM-L6-v2')

X_train_text = encoder.encode(
    X_train['text'].tolist(),
    show_progress_bar=True
)

X_test_text = encoder.encode(
    X_test['text'].tolist(),
    show_progress_bar=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/602 [00:00<?, ?it/s]

Batches:   0%|          | 0/258 [00:00<?, ?it/s]

In [19]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)

X_train_cat = ohe.fit_transform(
    X_train[['Age of User']]
)

X_test_cat = ohe.transform(
    X_test[['Age of User',]]
)

In [20]:
import numpy as np

X_train_final = np.hstack([
    X_train_text,
    X_train_cat
])

X_test_final = np.hstack([
    X_test_text,
    X_test_cat
])

In [21]:
X_test_final

array([[ 0.0843192 , -0.04302008,  0.03828492, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.08512202, -0.02040928,  0.06572263, ...,  0.        ,
         1.        ,  0.        ],
       [-0.02718352,  0.07542134,  0.02589726, ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [-0.06751353,  0.04104044,  0.00271727, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.05175971, -0.02320775,  0.09239133, ...,  0.        ,
         0.        ,  0.        ],
       [-0.00360848,  0.02759264,  0.02608009, ...,  0.        ,
         0.        ,  0.        ]], shape=(8244, 390))

In [22]:
pd.Series(y_train).value_counts()

1    7782
2    6007
0    5447
Name: count, dtype: int64

In [23]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score

lr = LogisticRegression(max_iter=5000)
lr.fit(X_train_final, y_train)

y_pred = lr.predict(X_test_final)
y_pred_train = lr.predict(X_train_final)

print('train')
print(accuracy_score(y_train, y_pred_train))
print(f1_score(y_train, y_pred_train, average='weighted'))
print(confusion_matrix(y_train, y_pred_train))
print(classification_report(y_train, y_pred_train))

print('test')
print(accuracy_score(y_test, y_pred))
print(f1_score(y_test, y_pred, average='weighted'))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

train
0.706487835308796
0.7071346658305669
[[3645 1572  230]
 [1156 5594 1032]
 [ 327 1329 4351]]
              precision    recall  f1-score   support

           0       0.71      0.67      0.69      5447
           1       0.66      0.72      0.69      7782
           2       0.78      0.72      0.75      6007

    accuracy                           0.71     19236
   macro avg       0.71      0.70      0.71     19236
weighted avg       0.71      0.71      0.71     19236

test
0.6885007278020379
0.6893642377815695
[[1566  660  108]
 [ 562 2315  458]
 [ 136  644 1795]]
              precision    recall  f1-score   support

           0       0.69      0.67      0.68      2334
           1       0.64      0.69      0.67      3335
           2       0.76      0.70      0.73      2575

    accuracy                           0.69      8244
   macro avg       0.70      0.69      0.69      8244
weighted avg       0.69      0.69      0.69      8244



In [24]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=5000, penalty='l2',solver='lbfgs')
lr.fit(X_train_final, y_train)

y_pred = lr.predict(X_test_final)
y_pred_train = lr.predict(X_train_final)

print('train')
print(accuracy_score(y_train, y_pred_train))
print(f1_score(y_train, y_pred_train, average='weighted'))
print(confusion_matrix(y_train, y_pred_train))
print(classification_report(y_train, y_pred_train))

print('test')
print(accuracy_score(y_test, y_pred))
print(f1_score(y_test, y_pred, average='weighted'))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

/Users/hemantpatidar/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


train
0.706487835308796
0.7071346658305669
[[3645 1572  230]
 [1156 5594 1032]
 [ 327 1329 4351]]
              precision    recall  f1-score   support

           0       0.71      0.67      0.69      5447
           1       0.66      0.72      0.69      7782
           2       0.78      0.72      0.75      6007

    accuracy                           0.71     19236
   macro avg       0.71      0.70      0.71     19236
weighted avg       0.71      0.71      0.71     19236

test
0.6885007278020379
0.6893642377815695
[[1566  660  108]
 [ 562 2315  458]
 [ 136  644 1795]]
              precision    recall  f1-score   support

           0       0.69      0.67      0.68      2334
           1       0.64      0.69      0.67      3335
           2       0.76      0.70      0.73      2575

    accuracy                           0.69      8244
   macro avg       0.70      0.69      0.69      8244
weighted avg       0.69      0.69      0.69      8244



In [25]:
from sklearn.svm import LinearSVC

svc = LinearSVC()
svc.fit(X_train_final, y_train)

y_pred = svc.predict(X_test_final)
y_pred_train = svc.predict(X_train_final)

print('train')
print(accuracy_score(y_train, y_pred_train))
print(f1_score(y_train, y_pred_train, average='weighted'))
print(confusion_matrix(y_train, y_pred_train))
print(classification_report(y_train, y_pred_train))

print('test')
print(accuracy_score(y_test, y_pred))
print(f1_score(y_test, y_pred, average='weighted'))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

train
0.7065398211686421
0.706869250675229
[[3696 1471  280]
 [1218 5469 1095]
 [ 398 1183 4426]]
              precision    recall  f1-score   support

           0       0.70      0.68      0.69      5447
           1       0.67      0.70      0.69      7782
           2       0.76      0.74      0.75      6007

    accuracy                           0.71     19236
   macro avg       0.71      0.71      0.71     19236
weighted avg       0.71      0.71      0.71     19236

test
0.6866812227074236
0.6871023547469303
[[1605  603  126]
 [ 593 2234  508]
 [ 163  590 1822]]
              precision    recall  f1-score   support

           0       0.68      0.69      0.68      2334
           1       0.65      0.67      0.66      3335
           2       0.74      0.71      0.72      2575

    accuracy                           0.69      8244
   macro avg       0.69      0.69      0.69      8244
weighted avg       0.69      0.69      0.69      8244



In [26]:
from sklearn.naive_bayes import GaussianNB

naive = GaussianNB()
naive.fit(X_train_final, y_train)

y_pred = naive.predict(X_test_final)
y_pred_train = naive.predict(X_train_final)

print('train')
print(accuracy_score(y_train, y_pred_train))
print(f1_score(y_train, y_pred_train, average='weighted'))
print(confusion_matrix(y_train, y_pred_train))
print(classification_report(y_train, y_pred_train))

print('test')
print(accuracy_score(y_test, y_pred))
print(f1_score(y_test, y_pred, average='weighted'))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

train
0.6559055936785194
0.6566691945890801
[[3554 1576  317]
 [1510 5110 1162]
 [ 521 1533 3953]]
              precision    recall  f1-score   support

           0       0.64      0.65      0.64      5447
           1       0.62      0.66      0.64      7782
           2       0.73      0.66      0.69      6007

    accuracy                           0.66     19236
   macro avg       0.66      0.66      0.66     19236
weighted avg       0.66      0.66      0.66     19236

test
0.6517467248908297
0.652276231465708
[[1565  626  143]
 [ 662 2148  525]
 [ 217  698 1660]]
              precision    recall  f1-score   support

           0       0.64      0.67      0.66      2334
           1       0.62      0.64      0.63      3335
           2       0.71      0.64      0.68      2575

    accuracy                           0.65      8244
   macro avg       0.66      0.65      0.65      8244
weighted avg       0.65      0.65      0.65      8244



In [27]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=101, max_depth=3, n_jobs=3)
rf.fit(X_train_final, y_train)

y_pred = rf.predict(X_test_final)
y_pred_train = rf.predict(X_train_final)

print('train')
print(accuracy_score(y_train, y_pred_train))
print(f1_score(y_train, y_pred_train, average='weighted'))
print(confusion_matrix(y_train, y_pred_train))
print(classification_report(y_train, y_pred_train))

print('test')
print(accuracy_score(y_test, y_pred))
print(f1_score(y_test, y_pred, average='weighted'))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

train
0.4911624038261593
0.3916912393967223
[[  37 5373   37]
 [   3 7579  200]
 [   1 4174 1832]]
              precision    recall  f1-score   support

           0       0.90      0.01      0.01      5447
           1       0.44      0.97      0.61      7782
           2       0.89      0.30      0.45      6007

    accuracy                           0.49     19236
   macro avg       0.74      0.43      0.36     19236
weighted avg       0.71      0.49      0.39     19236

test
0.4835031538088307
0.38165327063645144
[[  12 2303   19]
 [   2 3238   95]
 [   1 1838  736]]
              precision    recall  f1-score   support

           0       0.80      0.01      0.01      2334
           1       0.44      0.97      0.60      3335
           2       0.87      0.29      0.43      2575

    accuracy                           0.48      8244
   macro avg       0.70      0.42      0.35      8244
weighted avg       0.67      0.48      0.38      8244

